# Get the price data

The Dukascopy export form on their website downloads **one day at a time**. Two
years would be 730 downloads by hand.

The same data sits behind a plain HTTP feed, one file per hour, and this
notebook pulls the whole range in one go — on Google's machines, so nothing is
installed on your Chromebook.

---

## Run it one cell at a time

Click into the first cell and press **Shift+Enter** to run it and move to the
next. Repeat. A few minutes of clicking, plus the download.

**Not Runtime → Run all.** Step 3 is a checkpoint: it fetches a single hour and
shows you the price it decoded, so you can confirm the feed is giving you what
you expect before Step 4 spends half an hour on the full range.

Run all would blow straight past that. Step 4 does refuse to start if Step 3
failed its own check — so Run all is *safe*, it just wastes your time when
something is wrong instead of telling you at once.

## Step 1 — Get the code

In [ ]:
import os, shutil, subprocess

REPO   = "https://github.com/dboy140/Dboytrades.git"
BRANCH = "claude/ict-nbbtrader-trading-system-43hipg"

if os.path.isdir("/content/Dboytrades"):
    shutil.rmtree("/content/Dboytrades")

r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPO, "/content/Dboytrades"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit(f"Could not download the code:\n{r.stderr}")

os.chdir("/content/Dboytrades")
%pip install -q --upgrade pydantic
print("Step 1 done.")

## Step 2 — Choose what to download

`EURUSD` and `GBPUSD` are the pairs the NBB rules are stated for. `XAUUSD` is
covered by neither source (GAPS G-04), so treat anything it produces as
exploratory.

Two years is the target. That is not a round number picked for neatness — the
trade windows are about an hour a day and a setup does not appear every day, so
a shorter file gives a *confident wrong* answer rather than a cautious one.

In [ ]:
from datetime import datetime, timezone

SYMBOL = "EURUSD"        # EURUSD | GBPUSD | XAUUSD | USDJPY ...
START  = datetime(2024, 1, 1, tzinfo=timezone.utc)
END    = datetime(2026, 1, 1, tzinfo=timezone.utc)
SIDE   = "bid"           # match the offer side you would trade

from bot.dukascopy import hours_between
n = len(list(hours_between(START, END)))
print(f"{SYMBOL} {SIDE}: {START.date()} -> {END.date()}")
print(f"{n:,} hourly files to fetch (weekend hours come back empty)")

## Step 3 — Check one hour first

**This is the checkpoint. Read its output before moving on.**

It fetches a **single** hour and prints what came back. If the feed has moved or
the format has changed, you find out in five seconds rather than forty minutes.

It also checks the decoded price is the right order of magnitude for your
instrument. That specific mistake — every price 100x or 1000x out — is the one
worth guarding against, because the file it produces is perfectly well-formed:
right number of bars, right timestamps, every price wrong. Nothing downstream
can spot it. The backtest would simply run and report confident numbers.

In [ ]:
from datetime import datetime, timezone
from bot.dukascopy import (fetch_hour, decode_bi5, hour_url, ticks_to_minutes,
                           looks_plausible)

probe = datetime(2024, 3, 5, 14, tzinfo=timezone.utc)   # a Tuesday, London pm
print("URL:", hour_url(SYMBOL, probe))

PROBE_OK = False
raw = fetch_hour(SYMBOL, probe)
print(f"bytes returned: {len(raw):,}\n")

if not raw:
    print("EMPTY -- that hour has no data.")
    print("Try a different weekday/hour, or the symbol may not exist on this")
    print("feed under that name. Do not continue to Step 4.")
else:
    ticks = decode_bi5(raw, SYMBOL, probe)
    rows = ticks_to_minutes(ticks, SIDE)
    t = ticks[0]
    print(f"ticks decoded : {len(ticks):,}")
    print(f"first tick    : {t.ts}")
    print(f"  bid {t.bid}   ask {t.ask}   spread {t.ask - t.bid:.5f}")
    print(f"1-minute bars in this hour: {len(rows)}")

    PROBE_OK, why = looks_plausible(SYMBOL, t.bid)
    print(f"\nprice check   : {'PASS' if PROBE_OK else 'FAIL'} -- {why}")

    if PROBE_OK:
        print("\nLooks right. Continue to Step 4.")
    else:
        print("\nSTOP. Do not run Step 4 -- every bar would be scaled the same")
        print("way. Tell me what this printed and I will fix the point factor.")

## Step 4 — Download the full range

Expect several minutes for two years. Progress prints as it goes.

This refuses to start unless Step 3 passed, so it cannot quietly build a file
with every price scaled wrong.

Hours when the market is shut come back empty and are skipped — cheaper and more
accurate than encoding a holiday calendar here.

In [ ]:
import time
from bot.dukascopy import fetch_range

if not globals().get("PROBE_OK"):
    raise SystemExit(
        "Step 3 has not passed. Run it and read its output first.\n"
        "If it failed the price check, do not work around this -- the file "
        "would be well-formed and wrong in every price.")

def show(done, total):
    print(f"  {done:,}/{total:,} hours ({done/total:.0%})")

t0 = time.time()
rows = fetch_range(SYMBOL, START, END, side=SIDE, workers=16, progress=show)
print(f"\n{len(rows):,} one-minute bars in {time.time() - t0:.0f}s")
if rows:
    print(f"from {rows[0][0]} to {rows[-1][0]}")

## Step 5 — Write the CSV

In [ ]:
import pathlib
from bot.dukascopy import rows_to_csv

pathlib.Path("data/bars").mkdir(parents=True, exist_ok=True)
name = f"{SYMBOL}_1m_{START.date()}_{END.date()}.csv"
CSV = f"data/bars/{name}"
pathlib.Path(CSV).write_text(rows_to_csv(rows))

mb = pathlib.Path(CSV).stat().st_size / 1e6
print(f"wrote {CSV}  ({mb:.1f} MB, {len(rows):,} bars)")

## Step 6 — Is the file any good?

Bars alone are not enough. This checks the timestamps, looks for duplicates and
gaps, and counts how many bars actually fall inside each trade window.

**The session coverage numbers are the ones that matter.** A file with a million
rows and zero bars in the Silver Bullet window cannot test `SB-001`, however big
it is.

The timezone check should report no drift — this feed is already UTC and the
timestamps carry their offset. That check exists for broker exports, which are
stamped in server time and would put every session window hours out.

In [ ]:
!python -m bot.inspect_data "$CSV" 

## Step 7 — Download it to your Chromebook

In [ ]:
try:
    from google.colab import files
    files.download(CSV)
except Exception as exc:
    print(f"Auto-download unavailable ({exc}).")
    print("Use the folder icon in the left sidebar and download it from data/bars/.")

---

## Then what

Two options:

1. **Keep going here.** Open `notebooks/validate_colab.ipynb`, upload this CSV
   at its Step 3, and run the walk-forward validation.
2. **Send it to me.** Attach the CSV in the chat and I will run it.

Either way, expect a `FAILED` verdict. That is the honest base rate for a
mechanical rule set on out-of-sample data, and the validator is built to say it
— it currently refuses a pure random walk, which the previous version did not.
A pass would be the surprise.